Imports of libraries to be used

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.differentiate import derivative
from matplotlib.ticker import MaxNLocator

: 

data given with their values

In [ ]:
x=np.array([[1.0,3.0],
            [4.0,10.0]])
y=np.array([5.0,6.0])
m=np.array([-1.0,2.0])
b=np.array([1.0,1.0])
alpha=0.01
n_steps=3

m_start, b_start = m.copy(), b.copy()

the forward function for yhat=mx+b

In [ ]:
def forward(x,m,b):
    return x @ m + b

print("Error with the starting m and b:", forward(x,m,b))

the error calculation and function

In [ ]:
def get_error(x,y,m,b):
    return forward(x,m,b) - y

print("Error with the starting m and b:", get_error(x,y,m,b))

In [ ]:
def get_cost(x,y,m,b):
    error = get_error(x,y,m,b)
    return np.mean(error ** 2)

the partial derivative function that shows how a function changes when one value is changed while others stay fixed

In [ ]:
def partial_derivative(f,point,i):
    def f_of_t(t):
        results=[]
        for t_value in np.ravel(t):
            trial = point.copy()
            trial[i] = t_value
            results.append(f(trial))
        return np.array(results).reshape(np.shape(t))
    return derivative(f_of_t, point[i]).df

the gradient where its the derivative of the cost with respect to m and b values

In [ ]:
def gradient(x,y,m,b):
    point=np.concatenate([m,b]).astype(float)
    cost = lambda p: get_cost(x,y,p[:2],p[2:])
    slopes = [partial_derivative(cost,point,i) for i in range(4)]
    d_m = np.array(slopes[:2])
    d_b = np.array(slopes[2:])
    return d_m, d_b

d_m, d_b = gradient(x,y,m,b)
print("d cost / d m from SciPy:", d_m)
print("d cost / d b from SciPy:", d_b)

the update we move m and b against the gradient

In [ ]:
def update(x, y):
    global m, b
    d_m, d_b = gradient(x, y, m, b)
    m = m - alpha * d_m
    b = b - alpha * d_b

the loop of updating each value after an iteration

In [ ]:
m, b = m_start.copy(), b_start.copy() 

m_history = [m.copy()]
b_history = [b.copy()]
cost_history = [get_cost(x, y, m, b)]
print(f"Start     error = {get_error(x, y, m, b)}   cost = {cost_history[0]:.4f}")

for i in range(n_steps):
    update(x, y)

    m_history.append(m.copy())
    b_history.append(b.copy())
    cost_history.append(get_cost(x, y, m, b))

    if n_steps <= 10 or (i + 1) % 10 == 0:
        print(f"Update {i+1}  m = {np.round(m, 4)}   b = {np.round(b, 4)}")
        print(f"          error = {np.round(get_error(x, y, m, b), 4)}   cost = {cost_history[-1]:.4f}")

In [ ]:
print("Final m:", m)
print("Final b:", b)
print("Final predictions:", forward(x, m, b))
print("Targets y:        ", y)
print("Final error:", get_error(x, y, m, b))
print("Final cost (MSE):", get_cost(x, y, m, b))